# Strategy Validation — Walk-Forward Analysis

Validates E1/E2 strategies using walk-forward backtesting results. Key questions:
1. **Does the strategy have positive EV out-of-sample?** (test PF > 1.0)
2. **Are we overfitting?** (train PF >> test PF)
3. **Which pairs should be in the ALLOW list?** (test PF ≥ 1.3)
4. **How does performance vary by market regime?**

Run this after `e1_walk_forward` / `e2_walk_forward` tasks complete.

In [ ]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import matplotlib.pyplot as plt
from helpers import (
    get_mongo, load_walk_forward, walk_forward_summary,
    load_pair_historical, load_backtest_trades, load_candidates,
    plot_equity_curve, plot_drawdown, plot_close_type_breakdown,
    plot_pnl_distribution, plot_train_vs_test,
)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
plt.style.use("seaborn-v0_8-whitegrid")

db = get_mongo()
print("Connected to MongoDB")

## 1. Walk-Forward Results — E1 Compression Breakout

In [ ]:
ENGINE = "E1"

# Load latest walk-forward results
wf = load_walk_forward(db, ENGINE)
if wf.empty:
    print(f"No walk-forward results for {ENGINE}. Run: python cli.py trigger-task --task e1_walk_forward --config hermes_pipeline.yml")
else:
    print(f"Walk-forward results: {len(wf)} rows, run_id={wf['run_id'].iloc[0]}")
    print(f"Folds: {wf['fold_index'].nunique()}, Pairs: {wf['pair'].nunique()}")
    print(f"Period types: {wf['period_type'].value_counts().to_dict()}")

    # Summary table
    summary = walk_forward_summary(wf)
    print(f"\n{'='*80}")
    print(f"WALK-FORWARD SUMMARY — {ENGINE}")
    print(f"{'='*80}")
    display(summary.style.format({
        "train_pf": "{:.2f}", "test_pf": "{:.2f}",
        "train_wr": "{:.1f}", "test_wr": "{:.1f}",
        "overfit_ratio": "{:.1f}x",
    }).background_gradient(subset=["test_pf"], cmap="RdYlGn", vmin=0.5, vmax=2.0))

In [ ]:
# Train vs Test scatter plot (overfitting check)
if not wf.empty:
    summary = walk_forward_summary(wf)
    plot_train_vs_test(summary, f"{ENGINE} — Train vs Test Profit Factor")
    plt.show()

    # Verdict distribution
    verdicts = summary.copy()
    verdicts["verdict"] = verdicts["test_pf"].apply(
        lambda x: "ALLOW" if x >= 1.3 else ("WATCH" if x >= 1.0 else "BLOCK")
    )
    print("\nVerdict distribution (from test PF):")
    print(verdicts["verdict"].value_counts())
    print(f"\nOverfit pairs (train/test ratio > 2x):")
    overfit = verdicts[verdicts["overfit_ratio"] > 2.0]
    if overfit.empty:
        print("  None")
    else:
        for _, row in overfit.iterrows():
            print(f"  {row['pair']}: train={row['train_pf']:.2f} test={row['test_pf']:.2f} ratio={row['overfit_ratio']:.1f}x")

## 2. Per-Fold Performance (Stability Check)

In [ ]:
# Per-fold aggregate PF (are results stable across time?)
if not wf.empty:
    fold_agg = wf.groupby(["fold_index", "period_type"]).agg(
        avg_pf=("profit_factor", "mean"),
        total_trades=("trades", "sum"),
        avg_wr=("win_rate", "mean"),
        total_pnl=("pnl_quote", "sum"),
    ).reset_index()

    print(f"Per-fold aggregate metrics:")
    display(fold_agg.style.format({
        "avg_pf": "{:.2f}", "avg_wr": "{:.1f}", "total_pnl": "{:.0f}",
    }))

    # Bar chart: test PF per fold
    test_folds = fold_agg[fold_agg["period_type"] == "test"]
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(test_folds["fold_index"], test_folds["avg_pf"],
                  color=["green" if pf >= 1.0 else "red" for pf in test_folds["avg_pf"]])
    ax.axhline(y=1.0, color="red", linestyle="--", alpha=0.5)
    ax.axhline(y=1.3, color="green", linestyle="--", alpha=0.5)
    ax.set_xlabel("Fold")
    ax.set_ylabel("Average Test PF")
    ax.set_title(f"{ENGINE} — Test-Period PF by Fold")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 3. Backtest Trade Analysis (Latest Bulk Backtest)

In [ ]:
# Load latest bulk backtest trades
trades = load_backtest_trades(db, ENGINE)
if trades.empty:
    print(f"No backtest trades for {ENGINE}")
else:
    print(f"Loaded {len(trades)} trades from bulk backtest")
    print(f"Run ID: {trades['run_id'].iloc[0]}")
    print(f"Pairs: {trades['pair'].nunique()}")
    print(f"\nPnL stats:")
    print(trades["net_pnl_quote"].describe().round(2))

    # Equity curve
    plot_equity_curve(trades, f"{ENGINE} — Equity Curve (All Pairs)")
    plt.show()

    # Drawdown
    plot_drawdown(trades, f"{ENGINE} — Drawdown")
    plt.show()

    # Close type breakdown
    plot_close_type_breakdown(trades, f"{ENGINE} — Close Types")
    plt.show()

    # PnL distribution
    plot_pnl_distribution(trades, f"{ENGINE} — PnL Distribution")
    plt.show()

## 4. Current Pair Allowlist vs Walk-Forward Verdicts

In [ ]:
# Compare current verdicts (from bulk backtest) with walk-forward verdicts
ph = load_pair_historical(db, ENGINE)
if not ph.empty:
    cols = ["pair", "verdict", "profit_factor", "win_rate", "trades",
            "wf_avg_test_pf", "wf_avg_train_pf", "wf_overfit", "wf_folds"]
    available_cols = [c for c in cols if c in ph.columns]
    display(ph[available_cols].sort_values("pair"))

    print(f"\nVerdict counts:")
    print(ph["verdict"].value_counts())

    if "wf_avg_test_pf" in ph.columns:
        print(f"\nPairs where bulk PF > 1.3 but walk-forward test PF < 1.0 (DANGER):")
        danger = ph[(ph["profit_factor"] >= 1.3) & (ph["wf_avg_test_pf"] < 1.0)]
        if danger.empty:
            print("  None")
        else:
            for _, row in danger.iterrows():
                print(f"  {row['pair']}: bulk PF={row['profit_factor']:.2f}, WF test PF={row['wf_avg_test_pf']:.2f}")
else:
    print(f"No pair_historical data for {ENGINE}")

## 5. Repeat for E2 (Change ENGINE above, or run below)

In [ ]:
# Quick E2 summary
wf2 = load_walk_forward(db, "E2")
if not wf2.empty:
    summary2 = walk_forward_summary(wf2)
    print(f"E2 Walk-Forward Summary ({len(summary2)} pairs):")
    display(summary2.head(20))
    plot_train_vs_test(summary2, "E2 — Train vs Test Profit Factor")
    plt.show()
else:
    print("No E2 walk-forward results yet. Run: python cli.py trigger-task --task e2_walk_forward --config hermes_pipeline.yml")

## 6. Live Performance (Paper Trading Results)

In [ ]:
# Live paper trading results
for eng in ["E1", "E2"]:
    resolved = load_candidates(db, engine=eng, disposition="RESOLVED_TESTNET")
    if resolved.empty:
        print(f"{eng}: No resolved trades")
        continue

    print(f"\n{'='*60}")
    print(f"{eng} PAPER TRADING RESULTS ({len(resolved)} trades)")
    print(f"{'='*60}")

    if "testnet_pnl" in resolved.columns:
        pnl = resolved["testnet_pnl"].dropna().astype(float)
        wins = (pnl > 0).sum()
        total = len(pnl)
        print(f"Win rate: {wins}/{total} ({wins/total*100:.0f}%)")
        print(f"Total PnL: {pnl.sum():.2f}")
        print(f"Avg win: {pnl[pnl > 0].mean():.2f}" if wins > 0 else "Avg win: N/A")
        print(f"Avg loss: {pnl[pnl <= 0].mean():.2f}" if (total - wins) > 0 else "Avg loss: N/A")

    if "testnet_close_type" in resolved.columns:
        print(f"\nClose types:")
        print(resolved["testnet_close_type"].value_counts())

    cols = ["pair", "direction", "testnet_pnl", "testnet_close_type", "testnet_fill_price", "testnet_exit_price"]
    available = [c for c in cols if c in resolved.columns]
    display(resolved[available].head(20))